# Database initialization

In [ ]:
from db_management_tools import DataLoader
dataset_dir = r"D:\AIC26"
data_loader = DataLoader(dataset_dir)

# Upload data

In [ ]:
data_loader.create_and_fill_database(
    "aic2026_upload_test",
    upload_batch_size=256,
    upload_num_workers=16
)

# Query initialization

In [2]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model
from peripheral_tools import NotebookDisplay

image_text_feature_extractor = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
text_text_feature_extractor = BGEM3Model(name="BAAI/bge-m3", device="cuda")

query_manager = QueryManager(
    image_text_feature_extractor=image_text_feature_extractor,
    text_text_feature_extractor=text_text_feature_extractor
)
display = NotebookDisplay(315, 5)

[INFO] Loading quantized google/siglip2-so400m-patch14-384
[INFO] google/siglip2-so400m-patch14-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
query = "Một đàn sư tử đang nghỉ ngơi và leo trèo trên các bục gỗ trong khu nuôi dưỡng, phía trước có bảng thông tin của London Zoo phục vụ công tác theo dõi và bảo tồn động vật.. Sau đó có cảnh hai nhân viên mặc áo xanh lá đang cân và ghi nhận số liệu của một con vật trong khuôn viên sở thú."
limit = 100
points = query_manager.aggregated_search_v2(
    "aic2026",
    query=query,
    search_limit=limit,
    aggregation_neighbor_limit=10,
    time_distance_threshold=30,
    use_time_distance=True,
    entropy_based_bonus=True,
    agg_z_score_sigmoid_norm=True,
    aggregation_weight=0.5,
    # log=True
)

# print("\nList of candidates:")
# for point in points: print(f"{point.payload["vid_id"]}_{point.payload["frame_id"]}")
# display.display(points)

c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\lab\db_code\db_management_tools.py:694: RuntimeWarning: invalid value encountered in scalar divide
  softmax_score = max(exp_score / np.sum(exp_score), eps)


In [ ]:
points = query_manager.get_points(
    "aic2026",
    filters=query_manager.create_filters(vid_id="L21_V001", time_order=(499.5, 499.5)),
)
print(len(points))
print(points[:5])

In [ ]:
import numpy as np
np.ptp(np.array([1, 0 , 4.4]))

# Benchmark

In [ ]:
from peripheral_tools import Benchmarker, BenchmarkerV2

test_settings = [
    # {
    #     "using_vectors": ["siglip2_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_sparse"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    # },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
]

benchmarker = BenchmarkerV2(query_manager)
res = benchmarker.do_benchmark(
    "aic26_mini",
    test_questions=r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\question_bank\questions_with_answers.csv",
    test_settings=test_settings,
    save_record=False
)